In [1]:
import numpy as np
import pandas as pd
import gymnasium as gym
import gym_trading_env
from stable_baselines3 import DQN

# Testing MFI Feature

In [2]:
#DYNAMIQUE FEATURE

import pandas_ta as ta

def preprocess(df):
    df.ta.mfi(length=14, append=True) 
    df['MFI_14'] = df['MFI_14'] / 100.0
    df.fillna(50, inplace=True)
    return df

df = preprocess(pd.read_pickle('./data/binance-ETHUSD-1h.pkl'))
df.head(5)

,open,high,low,close,volume,date_close,MFI_14
date_open,,,,,,,
2020-08-18 07:00:00,430.00,435.00,410.00,430.30,487.154463,2020-08-18 08:00:00,50.0
2020-08-18 08:00:00,430.27,431.79,430.27,430.80,454.176153,2020-08-18 09:00:00,50.0
2020-08-18 09:00:00,430.86,431.13,428.71,429.35,1183.710884,2020-08-18 10:00:00,50.0
2020-08-18 10:00:00,429.75,432.69,428.59,431.90,1686.183227,2020-08-18 11:00:00,50.0
2020-08-18 11:00:00,432.09,432.89,426.99,427.45,1980.692724,2020-08-18 12:00:00,50.0


# Environnement

In [3]:
def reward_function(history):   
    current_val = history["portfolio_valuation", -1]
    last_val = history["portfolio_valuation", -2]
    reward = (current_val / last_val) - 1
    
    return reward

base_env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess,
    portfolio_initial_value=1_000,
    trading_fees=0.1/100,
    borrow_interest_rate=0.02/100/24,
    reward_function=reward_function,
)

# Wrapper

In [4]:
from gym_trading_env.wrapper import DiscreteActionsWrapper

env = DiscreteActionsWrapper(base_env, positions=[-1, 0, 1, 2])
obs, _ = env.reset()
obs, reward, terminated, truncated, info = env.step(3)
print("obs :", obs)
print("reward :", reward)
print("terminated :", terminated)
print("truncated :", truncated)
print("info :", info)

obs : [2.       2.421428]
reward : -0.2973053255383884
terminated : False
truncated : False
info : {'idx': 1, 'step': 1, 'date': np.datetime64('2021-01-29T13:00:00.000000000'), 'position_index': 3, 'position': 2, 'real_position': np.float64(2.421427858751112), 'data_MFI_14': 50.0, 'data_high': 0.05699, 'data_volume': 47022463.7, 'data_close': 0.04413, 'data_open': 0.05181, 'data_date_close': Timestamp('2021-01-29 14:00:00'), 'data_low': 0.042, 'portfolio_valuation': np.float64(702.6946744616116), 'portfolio_distribution_asset': np.float64(38557.09179554475), 'portfolio_distribution_fiat': 0, 'portfolio_distribution_borrowed_asset': 0, 'portfolio_distribution_borrowed_fiat': np.float64(998.8214629635869), 'portfolio_distribution_interest_asset': 0.0, 'portfolio_distribution_interest_fiat': np.float64(0.008323512191363224), 'reward': np.float64(-0.2973053255383884)}


# Evaluation

In [5]:
def metric_portfolio_valuation(history):
    return round(history['portfolio_valuation', -1], 2)

env.add_metric('Portfolio Valuation', metric_portfolio_valuation)

done = False
obs, _ = env.reset()

while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

AttributeError: 'DiscreteActionsWrapper' object has no attribute 'add_metric'

In [13]:
portfolio_valuation = env.historical_info['portfolio_valuation', -1]
# Si on avait WandB :
# run.summary['portfolio_valuation'] = portfolio_valuation
# On simule ça par un simple print...
print(portfolio_valuation)

0.005576465839266826


In [6]:
metrics = env.get_metrics()
print(metrics)
portfolio_valuation = metrics['Portfolio Valuation']
print(portfolio_valuation)

AttributeError: 'DiscreteActionsWrapper' object has no attribute 'get_metrics'

# Model

In [13]:
model = DQN(
    "MlpPolicy", 
    env, 
    verbose=1, 
    learning_rate=0.00001,
    buffer_size=100000,
    batch_size=32,
    exploration_fraction=0.2, # Explore during 50% of training
    exploration_initial_eps=1.0, # Start with 100% exploration
    exploration_final_eps=0.05,
)
model.learn(total_timesteps = 300000, log_interval = 4)
model.save("dqn_test")


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Market Return : 39.94%   |   Portfolio Return : -98.58%   |   
Market Return : 10.54%   |   Portfolio Return : -99.74%   |   
Market Return : 44.45%   |   Portfolio Return : -98.81%   |   
Market Return : 904.05%   |   Portfolio Return : -100.00%   |   
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.39e+04 |
|    ep_rew_mean      | -12.1    |
|    exploration_rate | 0.119    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 998      |
|    time_elapsed     | 55       |
|    total_timesteps  | 55671    |
| train/              |          |
|    learning_rate    | 1e-05    |
|    loss             | 3.65e-05 |
|    n_updates        | 13892    |
----------------------------------
Market Return :  5.27%   |   Portfolio Return : -74.63%   |   
Market Return : 27.70%   |   Portfolio Return : -0.15%   |   
Ma

In [14]:
nb_episodes = 10
for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    print(f'Episode n˚{episode}')
    done = False

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(int(action))
        done = terminated or truncated

    if terminated:
        print('Argent perdu')
    elif truncated:
        print('Épisode terminé')
    env.unwrapped.save_for_render(dir = "render_logs")



Episode n˚1
Market Return : 103.26%   |   Portfolio Return : 103.18%   |   
Épisode terminé
Episode n˚2
Market Return : 10.54%   |   Portfolio Return : 10.51%   |   
Épisode terminé
Episode n˚3
Market Return : 39.94%   |   Portfolio Return : -0.01%   |   
Épisode terminé
Episode n˚4
Market Return : 677.81%   |   Portfolio Return : 677.44%   |   
Épisode terminé
Episode n˚5
Market Return : 103.26%   |   Portfolio Return : -0.00%   |   
Épisode terminé
Episode n˚6
Market Return : 44.45%   |   Portfolio Return : -0.00%   |   
Épisode terminé
Episode n˚7
Market Return : 904.05%   |   Portfolio Return : 903.83%   |   
Épisode terminé
Episode n˚8
Market Return : 163.91%   |   Portfolio Return : -0.01%   |   
Épisode terminé
Episode n˚9
Market Return : 27.70%   |   Portfolio Return : -0.00%   |   
Épisode terminé
Episode n˚10
Market Return :  5.27%   |   Portfolio Return :  5.26%   |   
Épisode terminé
